# 🌫️ AQI Prediction Model — Full Pipeline from Scratch
**City:** Lahore | **Data:** MongoDB Atlas | **Goal:** Predict AQI value + Category

### Pipeline:
1. Load data from MongoDB
2. EDA & Visualization
3. Feature Engineering
4. Train Multiple Models
5. Compare & Select Best Model
6. Save Model

## 📦 Step 1 — Install & Import Libraries

In [ ]:
# Install required packages
!pip install pymongo[srv] pandas numpy scikit-learn xgboost lightgbm matplotlib seaborn joblib python-dotenv --quiet

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from dotenv import load_dotenv
load_dotenv()

from pymongo import MongoClient
from pathlib import Path

# Models
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Utilities
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import joblib

print('✅ All libraries imported!')
plt.style.use('seaborn-v0_8-darkgrid')

## 🗄️ Step 2 — Load Data from MongoDB

In [ ]:
MONGODB_URI = os.getenv('MONGODB_URI', '')

if not MONGODB_URI:
    raise ValueError('❌ MONGODB_URI not set! Add it to your .env file.')

# Connect to MongoDB
client = MongoClient(MONGODB_URI)
db     = client['aqi_pipeline']
col    = db['weather_aqi_features']

# Load all documents
docs = list(col.find({}, {'_id': 0}))  # exclude _id
df   = pd.DataFrame(docs)
client.close()

print(f'✅ Loaded {len(df)} rows from MongoDB')
print(f'📊 Columns: {df.shape[1]}')
df.head(3)

## 🔍 Step 3 — EDA (Exploratory Data Analysis)

In [ ]:
print('=== Dataset Info ===')
print(f'Shape: {df.shape}')
print(f'\nData Types:\n{df.dtypes.value_counts()}')
print(f'\nMissing Values (top 15):')
print(df.isnull().sum().sort_values(ascending=False).head(15))

In [ ]:
# AQI Distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogram
axes[0].hist(df['aqi'].dropna(), bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('AQI Distribution', fontsize=14)
axes[0].set_xlabel('AQI Value')

# AQI Category counts
if 'aqi_cat_label' in df.columns:
    df['aqi_cat_label'].value_counts().plot(kind='bar', ax=axes[1], color='coral', edgecolor='white')
    axes[1].set_title('AQI Category Distribution', fontsize=14)
    axes[1].tick_params(axis='x', rotation=45)

# AQI over time
if 'timestamp' in df.columns:
    df_sorted = df.sort_values('timestamp')
    axes[2].plot(range(len(df_sorted)), df_sorted['aqi'].values, alpha=0.6, color='green', linewidth=0.8)
    axes[2].set_title('AQI Over Time', fontsize=14)
    axes[2].set_xlabel('Time Index')
    axes[2].set_ylabel('AQI')

plt.tight_layout()
plt.savefig('aqi_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA plots saved!')

In [ ]:
# Correlation heatmap
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[numeric_cols].corr()

# Top features correlated with AQI
if 'aqi' in corr.columns:
    top_corr = corr['aqi'].abs().sort_values(ascending=False).head(20)
    print('Top 20 features correlated with AQI:')
    print(top_corr)

    plt.figure(figsize=(10, 6))
    top_corr[1:].plot(kind='barh', color='teal')
    plt.title('Feature Correlation with AQI', fontsize=14)
    plt.xlabel('Absolute Correlation')
    plt.tight_layout()
    plt.savefig('feature_correlation.png', dpi=150, bbox_inches='tight')
    plt.show()

## ⚙️ Step 4 — Feature Engineering & Preprocessing

In [ ]:
df_model = df.copy()

# Parse timestamp
if 'timestamp' in df_model.columns:
    df_model['timestamp'] = pd.to_datetime(df_model['timestamp'], utc=True, errors='coerce')
    df_model['hour']        = df_model['timestamp'].dt.hour
    df_model['day_of_week'] = df_model['timestamp'].dt.dayofweek
    df_model['month']       = df_model['timestamp'].dt.month
    df_model['day_of_year'] = df_model['timestamp'].dt.dayofyear
    df_model['is_weekend']  = (df_model['day_of_week'] >= 5).astype(int)
    df_model['is_night']    = ((df_model['hour'] >= 22) | (df_model['hour'] <= 6)).astype(int)
    df_model = df_model.sort_values('timestamp').reset_index(drop=True)

# Encode city
if 'city' in df_model.columns:
    le = LabelEncoder()
    df_model['city_encoded'] = le.fit_transform(df_model['city'].astype(str))

# Drop non-feature columns
drop_cols = ['timestamp', 'city', 'dominant_poll', 'aqi_cat_label', 'aqi_cat_ordinal']
drop_cols = [c for c in drop_cols if c in df_model.columns]
df_model  = df_model.drop(columns=drop_cols)

print(f'✅ Features after engineering: {df_model.shape[1]}')
print(f'Remaining columns: {list(df_model.columns)[:10]}...')

In [ ]:
# Define target and features
TARGET = 'aqi'

# Remove rows where target is null
df_model = df_model.dropna(subset=[TARGET])

# Select only numeric columns
feature_cols = [c for c in df_model.select_dtypes(include=[np.number]).columns if c != TARGET]

X = df_model[feature_cols]
y = df_model[TARGET]

print(f'✅ X shape: {X.shape}')
print(f'✅ y shape: {y.shape}')
print(f'\nTarget stats:')
print(y.describe())

In [ ]:
# Train/test split (time-based — no shuffle for time series)
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f'✅ Train size: {len(X_train)} | Test size: {len(X_test)}')

# Impute missing values
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)
X_test_imp  = imputer.transform(X_test)

# Scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_imp)
X_test_sc  = scaler.transform(X_test_imp)

print('✅ Imputation and scaling done!')

## 🤖 Step 5 — Train Multiple Models & Compare

In [ ]:
models = {
    'Linear Regression' : LinearRegression(),
    'Ridge'             : Ridge(alpha=1.0),
    'Random Forest'     : RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting' : GradientBoostingRegressor(n_estimators=100, random_state=42),
    'XGBoost'           : XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'LightGBM'          : LGBMRegressor(n_estimators=100, random_state=42, verbose=-1),
}

results = []

for name, model in models.items():
    # Use scaled data for linear models, raw for tree models
    if name in ['Linear Regression', 'Ridge', 'SVR']:
        Xtr, Xte = X_train_sc, X_test_sc
    else:
        Xtr, Xte = X_train_imp, X_test_imp

    model.fit(Xtr, y_train)
    preds = model.predict(Xte)

    mae  = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2   = r2_score(y_test, preds)

    results.append({'Model': name, 'MAE': round(mae,2), 'RMSE': round(rmse,2), 'R2': round(r2,4)})
    print(f'{name:25s} → MAE: {mae:.2f} | RMSE: {rmse:.2f} | R²: {r2:.4f}')

results_df = pd.DataFrame(results).sort_values('R2', ascending=False)
print('\n=== Model Comparison ===')
print(results_df.to_string(index=False))

In [ ]:
# Plot model comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, metric in enumerate(['MAE', 'RMSE', 'R2']):
    data = results_df.sort_values(metric, ascending=(metric != 'R2'))
    colors = ['green' if i == 0 else 'steelblue' for i in range(len(data))]
    axes[i].barh(data['Model'], data[metric], color=colors, edgecolor='white')
    axes[i].set_title(f'Model {metric}', fontsize=13)
    axes[i].set_xlabel(metric)

plt.suptitle('Model Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 🏆 Step 6 — Tune Best Model (XGBoost)

In [ ]:
# Tune XGBoost (best model usually)
param_grid = {
    'n_estimators'   : [100, 200, 300],
    'max_depth'      : [3, 5, 7],
    'learning_rate'  : [0.05, 0.1, 0.2],
    'subsample'      : [0.8, 1.0],
}

xgb = XGBRegressor(random_state=42, verbosity=0)

grid = GridSearchCV(
    xgb, param_grid,
    cv=3, scoring='r2',
    n_jobs=-1, verbose=1
)

grid.fit(X_train_imp, y_train)

print(f'\n✅ Best params: {grid.best_params_}')
print(f'✅ Best CV R²: {grid.best_score_:.4f}')

In [ ]:
# Evaluate tuned model
best_model = grid.best_estimator_
preds      = best_model.predict(X_test_imp)

mae  = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2   = r2_score(y_test, preds)

print(f'=== Tuned XGBoost Performance ===')
print(f'MAE  : {mae:.2f}')
print(f'RMSE : {rmse:.2f}')
print(f'R²   : {r2:.4f}')

# Actual vs Predicted plot
plt.figure(figsize=(10, 5))
plt.plot(y_test.values[:200], label='Actual', alpha=0.8)
plt.plot(preds[:200],         label='Predicted', alpha=0.8)
plt.title('Actual vs Predicted AQI (first 200 test samples)', fontsize=13)
plt.xlabel('Sample')
plt.ylabel('AQI')
plt.legend()
plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance
importance = pd.Series(best_model.feature_importances_, index=feature_cols)
top20 = importance.sort_values(ascending=False).head(20)

plt.figure(figsize=(10, 7))
top20.plot(kind='barh', color='darkorange')
plt.title('Top 20 Feature Importances', fontsize=13)
plt.xlabel('Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 10 features:')
print(top20.head(10))

## 💾 Step 7 — Save Model & Artifacts

In [ ]:
output_dir = Path('aqi_model_outputs')
output_dir.mkdir(exist_ok=True)

# Save model
joblib.dump(best_model, output_dir / 'aqi_xgb_model.pkl')
joblib.dump(imputer,    output_dir / 'imputer.pkl')
joblib.dump(scaler,     output_dir / 'scaler.pkl')

# Save feature list
import json
with open(output_dir / 'feature_cols.json', 'w') as f:
    json.dump(feature_cols, f)

# Save results
results_df.to_csv(output_dir / 'model_comparison.csv', index=False)

print('✅ Model saved to aqi_model_outputs/')
print('Files saved:')
for f in output_dir.iterdir():
    print(f'   {f.name}')

## ✅ Summary

In [ ]:
print('='*50)
print('       AQI MODEL TRAINING COMPLETE')
print('='*50)
print(f'Total records used  : {len(df_model)}')
print(f'Features used       : {len(feature_cols)}')
print(f'Best model          : XGBoost (tuned)')
print(f'MAE                 : {mae:.2f}')
print(f'RMSE                : {rmse:.2f}')
print(f'R²                  : {r2:.4f}')
print('='*50)
print('\nModel files saved in: aqi_model_outputs/')
print('\nNext steps:')
print('  1. Use model for real-time AQI prediction')
print('  2. Build a dashboard')
print('  3. Deploy as an API')